# Entraîner un modèle qui reconnaît un aliment (Phase 1)

Ce notebook entraîne, **automatiquement**, un modèle qui identifie **un aliment centré** sur une photo, puis l'exporte en `.tflite` pour l'app SmartCart.

**Comment l'utiliser :** clique ▶ sur chaque cellule, dans l'ordre, du haut vers le bas. Attends que chacune finisse (un ✅ ou un chiffre apparaît à gauche) avant la suivante.

**Avant de commencer**, active le GPU gratuit : menu *Exécution → Modifier le type d'exécution → Accélérateur matériel : GPU → Enregistrer*.

Les notions (données, entraînement, surapprentissage, transfer learning, export) sont expliquées dans `GUIDE.md`.

## 1. Vérifier l'environnement
On confirme que TensorFlow est là et que le GPU est actif (sinon l'entraînement sera lent mais marchera quand même).

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU détecté :', 'OUI ✅' if tf.config.list_physical_devices('GPU') else 'NON (ça marchera, mais plus lentement)')

## 2. Récupérer les données (images d'aliments déjà étiquetées)
On télécharge **Fruits-360**, un dataset public et gratuit : des dizaines de milliers de photos de fruits et légumes, rangées par dossier (le nom du dossier = l'étiquette). Aucune inscription nécessaire.

C'est l'étape « données » du GUIDE : on réutilise un jeu déjà étiqueté pour éviter le travail le plus long.

In [ ]:
# Clone léger (--depth 1 = sans l'historique, plus rapide)
![ -d Fruit-Images-Dataset ] || git clone --depth 1 https://github.com/Horea94/Fruit-Images-Dataset.git

TRAIN_DIR = 'Fruit-Images-Dataset/Training'
TEST_DIR  = 'Fruit-Images-Dataset/Test'

import os
print(len(os.listdir(TRAIN_DIR)), 'catégories disponibles, par exemple :')
print(sorted(os.listdir(TRAIN_DIR))[:15])

## 3. Charger les images en jeux d'entraînement / validation / test
On découpe comme expliqué dans le GUIDE (§3) :
- **entraînement** : le modèle apprend dessus ;
- **validation** : pour vérifier qu'il progresse vraiment (images jamais apprises) ;
- **test** : le verdict final honnête.

Les images sont redimensionnées en 224×224 (la taille attendue par le modèle de base).

> Astuce : pour un modèle plus léger et plus rapide, tu peux ne garder que certaines catégories en passant `class_names=['Banana','Cucumber Ripe', ...]` (noms EXACTS des dossiers) à la première fonction. Par défaut, on prend **toutes** les catégories.

In [ ]:
IMG = (224, 224)
BATCH = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.15, subset='training', seed=42,
    image_size=IMG, batch_size=BATCH)

# On mémorise la liste des catégories MAINTENANT (perdue après optimisation).
class_names = train_ds.class_names
print(len(class_names), 'catégories à apprendre')

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=0.15, subset='validation', seed=42,
    image_size=IMG, batch_size=BATCH)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG, batch_size=BATCH, shuffle=False)

# Prélecture : accélère l'entraînement en préparant les images à l'avance.
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 4. Data augmentation (anti-surapprentissage)
On crée des variantes des images à la volée (miroir, rotation, zoom). Le modèle voit « le même objet » sous plein d'angles → il apprend l'objet, pas une photo précise. Voir GUIDE §5.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name='augmentation')

## 5. Construire le modèle (transfer learning)
On part de **MobileNetV2**, déjà entraîné sur des millions d'images : il « sait voir » contours, textures, formes. On **gèle** ce savoir (`trainable = False`) et on ajoute seulement une petite couche finale qui décide *quel aliment*. Voir GUIDE §6.

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=IMG + (3,), include_top=False, weights='imagenet')
base.trainable = False  # on garde le savoir pré-appris, on ne le ré-entraîne pas

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMG + (3,)),
    data_augmentation,
    # MobileNetV2 attend des pixels entre -1 et 1 (les nôtres sont 0..255).
    tf.keras.layers.Rescaling(1. / 127.5, offset=-1),
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),  # anti-surapprentissage
    tf.keras.layers.Dense(len(class_names), activation='softmax'),
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 6. Entraîner
C'est l'étape clé : le modèle regarde les images et **apprend**. À chaque *epoch* (passage complet), la précision monte. On surveille `accuracy` (entraînement) et `val_accuracy` (validation).

⏱️ Quelques minutes sur GPU. 5 epochs suffisent pour commencer ; tu pourras en remettre plus tard.

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

## 7. Évaluer honnêtement
On regarde les courbes (entraînement vs validation : si elles divergent = surapprentissage, GUIDE §5) puis le score sur le jeu de **test** jamais vu.

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'], label='entraînement')
plt.plot(history.history['val_accuracy'], label='validation')
plt.xlabel('epoch'); plt.ylabel('précision'); plt.legend(); plt.grid(True)
plt.show()

perte, precision = model.evaluate(test_ds, verbose=0)
print(f'Précision sur le jeu de test : {precision*100:.1f}%')

## 8. Tester sur TES propres photos (le vrai test)
Le meilleur test = tes photos, prises avec ton téléphone. Lance la cellule, choisis une image d'un fruit/légume, et vois si le modèle trouve juste.

In [ ]:
from google.colab import files
import numpy as np

envoi = files.upload()  # choisis une ou plusieurs images
for nom in envoi:
    img = tf.keras.utils.load_img(nom, target_size=IMG)
    arr = tf.keras.utils.img_to_array(img)[None, ...]
    proba = model.predict(arr, verbose=0)[0]
    i = int(np.argmax(proba))
    print(f'{nom} → {class_names[i]}  ({proba[i]*100:.1f}%)')

## 9. Exporter pour le téléphone (`.tflite`)
On convertit le modèle en TensorFlow Lite (compact, optimisé mobile) et on écrit la liste des étiquettes. Ce sont les **deux fichiers** à intégrer dans l'app. Voir GUIDE §8.

In [ ]:
convertisseur = tf.lite.TFLiteConverter.from_keras_model(model)
convertisseur.optimizations = [tf.lite.Optimize.DEFAULT]  # quantization : + léger
tflite = convertisseur.convert()

with open('aliments.tflite', 'wb') as f:
    f.write(tflite)
with open('labels.txt', 'w') as f:
    f.write('\n'.join(class_names))

print(f'Modèle exporté : {len(tflite)/1_000_000:.1f} Mo, {len(class_names)} étiquettes')

from google.colab import files
files.download('aliments.tflite')
files.download('labels.txt')

## Et après ?
Tu as `aliments.tflite` + `labels.txt`. Étape suivante (côté app) : un écran « scanner un aliment » qui prend une photo, la passe au modèle via `tflite_flutter`, et propose le nom trouvé.

**Pour améliorer** (le cycle du ML, GUIDE §9) : ajoute des photos réalistes, remets des epochs, ajuste, ré-exporte.

**Phase 2** (plusieurs aliments sur une photo) = détection d'objets, un problème différent : voir GUIDE §10.